In [0]:
# ====================================================================
# SILVER LAYER - STEP 2: CREATE ENRICHED ORDERS FACT TABLE
# ====================================================================
# Purpose: Join all cleaned silver tables to create one master fact table
#          with complete order information including customers, products,
#          payments, reviews, and sellers
# ====================================================================

from pyspark.sql.functions import (
    col, sum, avg, count, datediff, when, coalesce, 
    current_timestamp, round, first, max, min
)

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
SILVER_SCHEMA = f"{PROJECT_NAME}_silver"

print("=" * 80)
print("🔗 SILVER LAYER - JOIN ENRICHED ORDERS")
print("=" * 80)
print(f"Source: {CATALOG}.{SILVER_SCHEMA}")
print(f"Target: {CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# LOAD ALL CLEANED SILVER TABLES
# ====================================================================
print("📂 Loading silver tables...\n")

# Load each silver table
orders_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders")
print(f"✅ silver_orders: {orders_df.count():,} rows")

customers_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers")
print(f"✅ silver_customers: {customers_df.count():,} rows")

order_items_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items")
print(f"✅ silver_order_items: {order_items_df.count():,} rows")

payments_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_order_payments")
print(f"✅ silver_order_payments: {payments_df.count():,} rows")

reviews_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_order_reviews")
print(f"✅ silver_order_reviews: {reviews_df.count():,} rows")

products_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_products")
print(f"✅ silver_products: {products_df.count():,} rows")

sellers_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_sellers")
print(f"✅ silver_sellers: {sellers_df.count():,} rows")

category_translation_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_category_translation")
print(f"✅ silver_category_translation: {category_translation_df.count():,} rows")

print("\n" + "=" * 80 + "\n")

In [0]:
# ====================================================================
# STEP 1: AGGREGATE PAYMENTS PER ORDER
# ====================================================================
# Orders can have multiple payment methods, so we aggregate them first
print("💳 Aggregating payments per order...\n")

payments_agg_df = (payments_df
    .groupBy("order_id")
    .agg(
        sum("payment_value").alias("total_payment_value"),
        first("payment_type").alias("primary_payment_type"),
        max("payment_installments").alias("max_installments"),
        count("*").alias("payment_method_count")
    )
)

print(f"✅ Aggregated payments: {payments_agg_df.count():,} orders")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# STEP 2: ENRICH PRODUCTS WITH ENGLISH CATEGORY NAMES
# ====================================================================
print("🔤 Translating product categories to English...\n")

# Join products with category translation
products_enriched_df = (products_df
    .join(
        category_translation_df,
        "product_category_name",
        "left"  # LEFT JOIN to keep products even if translation missing
    )
    .withColumn(
        "product_category_english",
        coalesce(col("product_category_name_english"), col("product_category_name"))
    )
    .select(
        "product_id",
        "product_category_name",
        "product_category_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_photos_qty",
        "product_name_length",
        "product_description_length"
    )
)

print(f"✅ Products with English categories: {products_enriched_df.count():,} products")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# STEP 3: ENRICH ORDER ITEMS WITH PRODUCT & SELLER INFO
# ====================================================================
print("🛍️ Enriching order items with product and seller information...\n")

order_items_enriched_df = (order_items_df
    # Join with products
    .join(
        products_enriched_df,
        "product_id",
        "inner"  # Only keep items with valid products
    )
    # Join with sellers
    .join(
        sellers_df.select(
            col("seller_id"),
            col("seller_city").alias("seller_city"),
            col("seller_state").alias("seller_state"),
            col("seller_zip_code_prefix").alias("seller_zip_code")
        ),
        "seller_id",
        "inner"  # Only keep items with valid sellers
    )
)

print(f"✅ Order items enriched: {order_items_enriched_df.count():,} items")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# STEP 4: JOIN ORDERS WITH CUSTOMERS
# ====================================================================
print("👥 Joining orders with customer information...\n")

orders_with_customers_df = (orders_df
    .join(
        customers_df.select(
            col("customer_id"),
            col("customer_unique_id"),
            col("customer_zip_code_prefix").alias("customer_zip_code"),
            col("customer_city"),
            col("customer_state")
        ),
        "customer_id",
        "inner"  # Only keep orders with valid customers
    )
)

print(f"✅ Orders with customers: {orders_with_customers_df.count():,} orders")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# STEP 5: MASTER JOIN - COMBINE EVERYTHING
# ====================================================================
print("🔗 Creating master enriched orders table...\n")

enriched_orders_df = (orders_with_customers_df
    # Join with order items (this is the base - one row per item)
    .join(
        order_items_enriched_df,
        "order_id",
        "inner"
    )
    # Join with aggregated payments
    .join(
        payments_agg_df,
        "order_id",
        "left"  # LEFT JOIN because some orders might not have payment data
    )
    # Join with reviews
    .join(
        reviews_df.select(
            col("order_id"),
            col("review_score"),
            col("review_sentiment"),
            col("review_comment_message"),
            col("review_creation_date")
        ),
        "order_id",
        "left"  # LEFT JOIN because not all orders have reviews
    )
)

print(f"✅ Master join complete: {enriched_orders_df.count():,} rows")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# STEP 6: ADD CALCULATED BUSINESS METRICS
# ====================================================================
print("📊 Adding calculated business metrics...\n")

enriched_orders_df = (enriched_orders_df
    # Calculate order value per item (price + freight)
    .withColumn(
        "item_total_value",
        col("price") + col("freight_value")
    )
    
    # Calculate delivery time in days
    .withColumn(
        "actual_delivery_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    )
    
    # Calculate estimated delivery days
    .withColumn(
        "estimated_delivery_days",
        datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp"))
    )
    
    # Flag late deliveries
    .withColumn(
        "is_late_delivery",
        when(
            col("order_delivered_customer_date") > col("order_estimated_delivery_date"),
            True
        ).otherwise(False)
    )
    
    # Calculate delay in days (0 if on time or early)
    .withColumn(
        "delivery_delay_days",
        when(
            col("is_late_delivery") == True,
            datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))
        ).otherwise(0)
    )
    
    # Flag high-value items (above 100 BRL)
    .withColumn(
        "is_high_value_item",
        when(col("item_total_value") >= 100, True).otherwise(False)
    )
    
    # Extract year and month from purchase date
    .withColumn("order_year", year(col("order_purchase_timestamp")))
    .withColumn("order_month", month(col("order_purchase_timestamp")))
    .withColumn("order_quarter", quarter(col("order_purchase_timestamp")))
    
    # Add audit columns
    .withColumn("_enriched_at", current_timestamp())
)

print("✅ Calculated metrics added")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# STEP 7: SELECT FINAL COLUMNS & WRITE
# ====================================================================
print("💾 Writing enriched orders table...\n")

# Select and organize columns in logical order
final_df = enriched_orders_df.select(
    # Order identifiers
    "order_id",
    "order_item_id",
    
    # Customer information
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code",
    
    # Product information
    "product_id",
    "product_category_name",
    "product_category_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    
    # Seller information
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code",
    
    # Order details
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    
    # Pricing
    "price",
    "freight_value",
    "item_total_value",
    
    # Payment information
    "total_payment_value",
    "primary_payment_type",
    "max_installments",
    "payment_method_count",
    
    # Review information
    "review_score",
    "review_sentiment",
    "review_comment_message",
    
    # Calculated metrics
    "actual_delivery_days",
    "estimated_delivery_days",
    "is_late_delivery",
    "delivery_delay_days",
    "is_high_value_item",
    "order_year",
    "order_month",
    "order_quarter",
    
    # Audit columns
    "_enriched_at"
)

# Write to silver schema
full_table_name = f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched"

(final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name))

final_count = final_df.count()
print(f"✅ Written to {full_table_name}")
print(f"✅ Total rows: {final_count:,}")
print(f"✅ Total columns: {len(final_df.columns)}")
print("\n" + "=" * 80 + "\n")

In [0]:
# ====================================================================
# DATA QUALITY SUMMARY
# ====================================================================
print("📊 DATA QUALITY SUMMARY\n")
print("=" * 80)

enriched_df = spark.table(full_table_name)

# Basic statistics
total_rows = enriched_df.count()
unique_orders = enriched_df.select("order_id").distinct().count()
unique_customers = enriched_df.select("customer_id").distinct().count()
unique_products = enriched_df.select("product_id").distinct().count()
unique_sellers = enriched_df.select("seller_id").distinct().count()

print(f"Total Rows (order items): {total_rows:,}")
print(f"Unique Orders: {unique_orders:,}")
print(f"Unique Customers: {unique_customers:,}")
print(f"Unique Products: {unique_products:,}")
print(f"Unique Sellers: {unique_sellers:,}")
print("-" * 80)

# Order status breakdown
print("\n📦 Order Status Distribution:")
enriched_df.groupBy("order_status").count().orderBy(col("count").desc()).show()

# Late delivery stats
late_delivery_count = enriched_df.filter(col("is_late_delivery") == True).count()
late_delivery_pct = (late_delivery_count / total_rows * 100)
print(f"Late Deliveries: {late_delivery_count:,} ({late_delivery_pct:.2f}%)")

# Payment type distribution
print("\n💳 Payment Type Distribution:")
enriched_df.groupBy("primary_payment_type").count().orderBy(col("count").desc()).show()

# Review score distribution
print("\n⭐ Review Score Distribution:")
enriched_df.filter(col("review_score").isNotNull()).groupBy("review_score").count().orderBy("review_score").show()

print("=" * 80)
print("\n✅ Silver Enriched Orders Table Complete!")
print("\n📝 Next Step: Run 03_create_dimension_tables.py to create customer/product/seller master tables\n")

In [0]:
%sql
-- Verify the table exists
SHOW TABLES IN workspace.retail_silver LIKE 'silver_orders_enriched';

In [0]:
%sql
-- Preview enriched orders
SELECT 
    order_id,
    customer_city,
    customer_state,
    product_category_english,
    price,
    freight_value,
    item_total_value,
    order_status,
    actual_delivery_days,
    is_late_delivery,
    review_score,
    primary_payment_type
FROM workspace.retail_silver.silver_orders_enriched
LIMIT 20;

In [0]:
# ====================================================================
# VERIFY SCHEMA
# ====================================================================
enriched_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")

print("\n📋 ENRICHED ORDERS SCHEMA\n")
print("=" * 80)
print(f"{'Column Name':<40} {'Data Type':<25}")
print("-" * 80)

for field in enriched_df.schema.fields:
    print(f"{field.name:<40} {str(field.dataType):<25}")

print("=" * 80)
print(f"\nTotal Columns: {len(enriched_df.columns)}")